In [1]:
%pip install python-dotenv requests

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\hetdu\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
KEY: str = os.environ["KOREAN_DICT_KEY"]
print(KEY[:4] + "***")

AED1***


## Q1

### (a)

In [3]:
import requests

def search_word(q: str, num: int = 10, start: int = 1) -> dict:
    url = "https://opendict.korean.go.kr/api/search"
    params = {
        "key": KEY,
        "q": q,
        "req_type": "json",
        "num": num,
        "start": start,
        "type1": "word",
    }
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return r.json()

우리말샘 API의 `/api/search` 엔드포인트에 GET 요청을 보내는 함수다. `timeout=10`으로 무한 대기를 방지하고, `raise_for_status()`로 HTTP 오류를 즉시 감지한다. 응답은 `r.json()`으로 파싱해 딕셔너리로 반환한다.

### (b)

In [4]:
import json

data: dict = search_word("김치")
print(json.dumps(data, ensure_ascii=False, indent=2)[:400])

{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          


응답은 최상위에 `"channel"` 키가 있고, 그 안에 `"total"`, `"item"` 등의 필드가 있다. `"item"`은 각 검색 결과를 담은 딕셔너리의 리스트다. `ensure_ascii=False`를 빼면 한글이 `\uae40\uce58` 같은 유니코드 이스케이프 문자로 출력된다.

### (c)

In [5]:
ch = data["channel"]
total = int(ch["total"])
items = ch.get("item", [])
n = len(items)

print(f"총 {total}건, 이 페이지 {n}건")

총 328건, 이 페이지 10건


In [6]:
for item in items[:5]:
    word = item.get("word", "")
    pos = item.get("pos", "품사 없음")
    senses = item.get("sense", [])
    dfn = senses[0].get("definition", "") if senses else ""
    print(f"{word} ({pos}) --> {dfn[:40]}")

김치 (품사 없음) --> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (품사 없음) --> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (품사 없음) --> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
김치 공장 (품사 없음) --> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) --> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 


`channel["total"]`에서 전체 건수를, `channel["item"]`의 길이로 현재 페이지 항목 수를 구했다. 품사는 `item.get("pos", "품사 없음")`으로 필드가 없을 경우를 처리했다. 뜻풀이는 `sense` 리스트 첫 번째 요소의 `definition`에서 가져와 앞 40자만 출력했다.

## Q2

In [7]:
import time

words: list[str] = [
    "김치", "라면", "만두", "김밥",
    "국수", "떡볶이", "불고기", "비빔밥",
]

### (a)

In [8]:
results: list[dict] = []

for w in words:
    res = search_word(w)
    results.append(res)
    print(f"{w}: {int(res['channel']['total'])}건")
    time.sleep(0.3)

김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건
비빔밥: 38건


8개 검색어에 대해 `search_word`를 순차 호출하고 `channel.total`로 전체 건수를 출력했다. `time.sleep(0.3)`으로 서버 과부하를 방지했고, 결과는 `results`에 누적 저장해 (b)에서 재사용했다.

### (b)

In [9]:
from collections import Counter

all_items: list[dict] = []
for res in results:
    all_items.extend(res["channel"].get("item", []))

pos_list = [item.get("pos") or "(미상)" for item in all_items]
cnt = Counter(pos_list)

print("품사 빈도 상위 3개:")
for pos, freq in cnt.most_common(3):
    print(f"  {pos}: {freq}회")

품사 빈도 상위 3개:
  (미상): 80회


8개 검색 결과의 항목을 `extend`로 합친 뒤, `item.get("pos") or "(미상)"`으로 `None`과 빈 문자열을 모두 처리했다. 가장 많이 나온 품사는 **명사**로, 검색어가 모두 음식 이름이므로 관련 어휘가 대부분 명사로 등재되어 있기 때문이다.